# Robust / Uncertain-input Classification

39番Notebookの標準binary GP contractを前提に、label noiseとcandidate-time input uncertaintyを比較します。


## 学習の目的

39番の標準GP分類を学んだ後に、**データの何が不確かなのか**を区別するNotebookです。ラベル（検査結果）を間違える問題と、指定した実験条件から実際の入力が揺れる問題は、異なるメカニズムです。

この例では0/1の合成ラベルを使用します。ラベルノイズモデルを構築していますが、デモの元データ自体に誤ラベルを人工注入しているわけではありません。

In [ ]:
import torch

from robotorchan.models.classification import (
    ContinuousUncertainInputBinarySingleTaskGPClassifier,
    LabelNoiseBinarySingleTaskGPClassifier,
)

torch.set_default_dtype(torch.double)
torch.manual_seed(0)


## 1. 小さな合成データ

### 形状を確認する

`train_X` は8点×1変数、`train_Y` は8個の0/1ラベル、`grid` は予測位置51点です。合成ラベルは `x > 0.5` のとき1になります。実務の測定ではこの正解関数は未知です。

In [ ]:
train_X = torch.tensor([[0.05], [0.15], [0.30], [0.48], [0.52], [0.70], [0.85], [0.95]])
train_Y = (train_X.squeeze(-1) > 0.5).to(dtype=torch.double)
grid = torch.linspace(0.0, 1.0, 51).unsqueeze(-1)


## 2. Label noise と input uncertainty

Label noiseは観測過程、input uncertaintyはcandidate入力の不確かさです。

### モデルの違い

- `LabelNoiseBinarySingleTaskGPClassifier` はラベル観測過程に関する頑健性を扱います。
- `ContinuousUncertainInputBinarySingleTaskGPClassifier` は入力の揺らぎを考慮した推論用APIを備えます。

両者は同じデータで初期化されますが、*不確実性の原因* を同一視していません。

In [ ]:
label_noise_model = LabelNoiseBinarySingleTaskGPClassifier(train_X, train_Y)
uncertain_model = ContinuousUncertainInputBinarySingleTaskGPClassifier(train_X, train_Y)
print(label_noise_model.classification_metadata)
print(uncertain_model.classification_metadata)


## 3. Candidate-time input uncertainty

### input_std と num_samples

`input_std=0.03` は連続入力の標準偏差を仮定します。入力 `x=0.5` であれば、実際の入力がその付近で揺らぐことを想定します。

`num_samples=32` は揺らいだ入力を扱う近似計算で用いるサンプル数です。有限サンプルのため予測は変動します。

**重要：** ここでの計算は学習前のモデルに対して行います。意味のある学習後の比較は次のセクションで行います。

In [ ]:
input_std = torch.full_like(grid, 0.03)
with torch.no_grad():
    probability = uncertain_model.predict_proba_with_input_uncertainty(
        grid, input_std=input_std, num_samples=32
    )
print(probability.shape)


## 4. 使い分け

- 誤ラベルはrobust likelihood側で扱います。
- candidate Xの揺らぎはuncertain-input側で扱います。
- Student-t regression likelihoodをbinary labelへ機械的に移植しません。
- categorical input uncertaintyには専用のexact marginalization APIがあります。

## 5. 学習・候補提案・ラベル評価

両モデルを既存の変分ELBOで学習し、入力不確実性を考慮した予測確率に基づいて、未ラベル候補を選択します。候補の選択は有限プールで行います。


### ELBO学習と候補選択の考え方

2つのモデルを同様の学習ループで更新し、入力の揺らぎを考慮した**クラス1予測確率**から二値エントロピーを求めています。エントロピーが大きい候補（ラベルが曖昧な候補）を未ラベル候補の有限集合から選びます。

ここでは **有限プールでの単純な最大値選択** であり、39番の `optimize_acqf` と同じ方法ではありません。

`observed_label` は合成関数から計算した疑似的な測定結果です。実際の設備では実験後にラベルが得られます。

In [ ]:
for model in (label_noise_model, uncertain_model):
    model.train()
    model.likelihood.train()
    mll = model.make_mll()
    optimizer = torch.optim.Adam(model.parameters(), lr=0.05)
    for _ in range(20):
        optimizer.zero_grad()
        loss = -mll(model.model(train_X), train_Y)
        loss.backward()
        optimizer.step()
    model.eval()
    model.likelihood.eval()

candidate_pool = torch.linspace(0.0, 1.0, 101).unsqueeze(-1)
with torch.no_grad():
    probabilities = uncertain_model.predict_proba_with_input_uncertainty(
        candidate_pool,
        input_std=torch.full_like(candidate_pool, 0.03),
        num_samples=32,
    )
    positive_probability = probabilities[..., 1]
    entropy = -(
        positive_probability.clamp(1e-8, 1 - 1e-8).log() * positive_probability
        + (1 - positive_probability).clamp(1e-8, 1 - 1e-8).log()
        * (1 - positive_probability)
    )
candidate = candidate_pool[entropy.argmax()].unsqueeze(0)
observed_label = (candidate.squeeze(-1) > 0.5).long()
print("candidate:", candidate, "observed label:", observed_label)


## 6. 図で比較する：通常の確率と入力の揺らぎを考慮した確率

学習済み `uncertain_model` のクラス1確率について、入力の揺らぎを考慮しない場合と考慮した場合を並べて表示します。

このグラフは**入力不確実性の効果を見るための説明図**です。ラベルノイズモデルとどちらが高性能かを検証する実験ではありません。

In [ ]:
import matplotlib.pyplot as plt

with torch.no_grad():
    base_probability = uncertain_model.predict_proba(candidate_pool)[..., 1]
    uncertain_probability = uncertain_model.predict_proba_with_input_uncertainty(
        candidate_pool,
        input_std=torch.full_like(candidate_pool, 0.03),
        num_samples=64,
    )[..., 1]

xs = candidate_pool.squeeze(-1).detach().cpu().numpy()
base_values = base_probability.detach().cpu().numpy()
uncertain_values = uncertain_probability.detach().cpu().numpy()
entropy_values = entropy.detach().cpu().numpy()
selected_x = candidate.detach().cpu().reshape(-1)[0].item()

fig, axes = plt.subplots(2, 1, figsize=(9, 7), sharex=True)
axes[0].plot(xs, base_values, label="Without input uncertainty")
axes[0].plot(xs, uncertain_values, label="With input uncertainty")
axes[0].scatter(
    train_X.squeeze(-1).detach().cpu().numpy(),
    train_Y.detach().cpu().numpy(),
    label="Observed labels",
)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_ylabel("P(y=1)")
axes[0].legend()
axes[1].plot(xs, entropy_values, label="Entropy (32 samples)")
axes[1].axvline(selected_x, linestyle="--", color="gray", label="Selected")
axes[1].set_ylabel("Entropy")
axes[1].set_xlabel("Input x")
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

### 図からの読み取り

1. 入力揺らぎを加えると、確率曲線がどこで変化するかを確認します。変化の向きや程度は学習結果に依存します。
2. エントロピー最大の場所は、必ずしも入力不確実性を考慮する前と同じとは限りません。
3. サンプル数を変えて再実行し、Monte Carlo誤差の影響を確認してみてください。

## 初学者向け補足：何が不確かなのかを区別する

同じ「不確実性」でも、原因が異なると対策も異なります。

| 状況 | 例 | 主な対処 |
|---|---|---|
| ラベルノイズ | 検査員ごとに合否判定が揺れる | ラベル生成・観測過程の頑健化 |
| 入力不確実性 | 設定温度と実温度にずれがある | 入力の揺らぎを考慮した予測 |
| モデル不確実性 | 未測定の領域が広い | 追加観測・Active Learning |

### Notebookの読み進め方

1. 合成データの0/1ラベルを確認します。
2. どのモデルがラベルノイズを扱い、どの処理が入力の揺らぎを扱うかを追います。
3. 学習後の予測確率を比較し、入力が揺れたときに候補の評価が変わるかを確認します。
4. 候補提案後に得られるラベルは実験・測定で初めて判明することを意識します。合成データの真値を使うのはデモ用です。

**注意：** ラベルノイズへの頑健性と、未知の入力変動への頑健性は同義ではありません。例の少ないepochや小さな合成データから、実問題での優劣を結論づけないでください。